# Task 2a: Corruption Classifier Validation & Benchmark Evaluation

Evaluates the trained corruption classifier on the deterministic test benchmark as required by the assignment.

### Mathematical Formulation
$$p = f(\tilde{x}) = [p_{clean},\, p_{salt},\, p_{blur},\, p_{occlusion}], \quad \hat{k} = \arg\max_{k}\, p_k$$

### Metrics Reported:
- **Overall Accuracy**
- **Macro-Averaged Precision, Recall, and F1-Score**
- **Per-Class Metrics Table** (clean, salt-and-pepper, Gaussian blur, rectangular occlusion)
- **Normalised 4-Class Confusion Matrix**
- **Accuracy Stratified by Severity Level** (low, medium, high)
- **Failure Mode Analysis** (misclassified examples grouped by confusion pair)

In [ ]:
# 1. Imports
import sys
from pathlib import Path
from collections import defaultdict
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix
)

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from src.device_utils import get_device
from src.models_classifier import CorruptionClassifier
from src.checkpoint_utils import load_checkpoint
from src.evaluation import PetBenchmarkDataset, image_quality_metrics

DEVICE = get_device()
CLASS_NAMES = ["clean", "salt", "blur", "occlusion"]


In [ ]:
# 2. Configuration – pick the Optuna-tuned checkpoint if present
FINAL_CKPT  = RESEARCH_ROOT / "checkpoints" / "task2a_classifier_final_best.pt"
SIMPLE_CKPT = RESEARCH_ROOT / "checkpoints" / "task2_classifier_best.pt"
CHECKPOINT_PATH = FINAL_CKPT if FINAL_CKPT.exists() else SIMPLE_CKPT


## 1. Load Trained Classifier

In [ ]:
# 3. Load model — infer architecture from checkpoint
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Train Task 2a first. Missing checkpoint: {CHECKPOINT_PATH}"
    )
saved  = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
state  = saved["model_state_dict"]
base_c = state["conv1.0.weight"].shape[0]
model  = CorruptionClassifier(base_channels=base_c).to(DEVICE)
ckpt   = load_checkpoint(CHECKPOINT_PATH, model, device=DEVICE)
model.eval()
print(f"Loaded epoch {ckpt.get('epoch')} – val_loss={ckpt.get('val_loss'):.4f}")


## 2. Quantitative Metrics on Deterministic Test Benchmark

In [ ]:
# 4. Run inference over the fixed test manifest
benchmark  = PetBenchmarkDataset(RESEARCH_ROOT / "test_manifest_official.json")
loader     = DataLoader(benchmark, batch_size=64, shuffle=False, num_workers=2)

y_true, y_pred, metadata = [], [], []

with torch.inference_mode():
    for corrupted, clean, labels, kinds, severities, names in loader:
        corrupted = corrupted.to(DEVICE)
        logits    = model(corrupted)
        preds     = logits.argmax(dim=1).cpu().tolist()
        y_true.extend(labels.tolist())
        y_pred.extend(preds)
        metadata.extend(list(zip(names, severities, kinds)))

# Macro metrics
acc = accuracy_score(y_true, y_pred)
prec, rec, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=list(range(4)), average=None, zero_division=0
)
macro_prec, macro_rec, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)
print(f"Overall accuracy : {acc*100:.2f}%")
print(f"Macro  precision : {macro_prec:.4f}")
print(f"Macro  recall    : {macro_rec:.4f}")
print(f"Macro  F1        : {macro_f1:.4f}")

per_class = pd.DataFrame({
    "class": CLASS_NAMES,
    "precision": prec.round(4),
    "recall": rec.round(4),
    "f1": f1.round(4),
    "support": support,
})
display(per_class)


## 3. Normalised Confusion Matrix

In [ ]:
# 5. Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=list(range(4)), normalize="true")
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    vmin=0, vmax=1, ax=ax
)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Normalised Confusion Matrix – Task 2a Classifier")
plt.tight_layout()
plt.show()


## 4. Accuracy by Corruption Severity

In [ ]:
# 6. Break down accuracy by severity level
sev_correct = defaultdict(int)
sev_total   = defaultdict(int)
for t, p, (_, severity, _) in zip(y_true, y_pred, metadata):
    sev_total[severity]   += 1
    sev_correct[severity] += int(t == p)

print("Accuracy by severity:")
for sev in sorted(sev_total):
    acc_s = sev_correct[sev] / sev_total[sev]
    print(f"  {sev:<8}: {acc_s*100:.2f}%  (n={sev_total[sev]})")


## 5. Failure Mode Analysis (Misclassified Examples)

In [ ]:
# 7. Show misclassified examples grouped by confusion pair
errors = [
    (CLASS_NAMES[t], CLASS_NAMES[p], sev, name)
    for t, p, (name, sev, _) in zip(y_true, y_pred, metadata)
    if t != p
]
if not errors:
    print("No misclassifications on this benchmark — classifier may be saturated.")
else:
    error_df = pd.DataFrame(errors, columns=["true", "predicted", "severity", "image"])
    print(f"Total misclassifications: {len(errors)} / {len(y_true)}")
    display(
        error_df.groupby(["true", "predicted", "severity"])
                .size()
                .reset_index(name="count")
                .sort_values("count", ascending=False)
                .head(12)
    )
